In [ ]:
!pip install -q ftfy beautifulsoup4 nltk umap-learn transformers accelerate sentencepiece joblib

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import re
import html
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from bs4 import BeautifulSoup
import ftfy

warnings.filterwarnings("ignore")
tqdm.pandas()

# ============================================================
# CHANGE THESE PATHS IF YOUR FILES ARE IN A DIFFERENT LOCATION
# ============================================================

CS_FILE = "/content/drive/MyDrive/cs_domain_main_cs_merged_abstracts.csv"
MED_FILE = "/content/drive/MyDrive/medi_domain_main_file_merged_abstracts_medi.csv"

BASE_OUTPUT_DIR = "/content/drive/MyDrive/cross_domain_tfidf_scibert_umap_outputs"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

STEP2_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "step2_preprocessing_outputs")
TFIDF_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "tfidf_umap_outputs")
SCIBERT_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "scibert_umap_outputs")
FINAL_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "final_outputs")

for d in [STEP2_OUTPUT_DIR, TFIDF_OUTPUT_DIR, SCIBERT_OUTPUT_DIR, FINAL_OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

RANDOM_STATE = 42

print("CS file exists:", os.path.exists(CS_FILE))
print("Medical file exists:", os.path.exists(MED_FILE))
print("Output folder:", BASE_OUTPUT_DIR)

In [ ]:
# ============================================================
# STEP 1 — Corpus Collection
# CS + Medical | AI vs Human
# ============================================================

def read_csv_safely(path):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None

    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"Loaded {os.path.basename(path)} using encoding: {enc}")
            return df
        except Exception as e:
            last_error = e
            print(f"Failed with {enc}: {e}")

    raise RuntimeError(f"Could not read file: {path}. Last error: {last_error}")

cs_df = read_csv_safely(CS_FILE)
med_df = read_csv_safely(MED_FILE)

cs_df = cs_df.drop(columns=[c for c in cs_df.columns if str(c).startswith("Unnamed")], errors="ignore")
med_df = med_df.drop(columns=[c for c in med_df.columns if str(c).startswith("Unnamed")], errors="ignore")

print("CS shape:", cs_df.shape)
print("CS columns:", cs_df.columns.tolist())
print()
print("Medical shape:", med_df.shape)
print("Medical columns:", med_df.columns.tolist())

# ============================================================
# Column names from your original notebook
# Change only if your CSV column names are different
# ============================================================

CS_TITLE_COL = "title"
CS_HUMAN_COL = "original_abstract"
CS_AI_COL = "ai_generated_abstract"

MED_TITLE_COL = "title"
MED_HUMAN_COL = "abstract"
MED_AI_COL = "ai_generated_abstract"

required_cs = [CS_TITLE_COL, CS_HUMAN_COL, CS_AI_COL]
required_med = [MED_TITLE_COL, MED_HUMAN_COL, MED_AI_COL]

for col in required_cs:
    if col not in cs_df.columns:
        raise ValueError(f"CS file missing column: {col}")

for col in required_med:
    if col not in med_df.columns:
        raise ValueError(f"Medical file missing column: {col}")

for col in required_cs:
    cs_df[col] = cs_df[col].fillna("").astype(str)

for col in required_med:
    med_df[col] = med_df[col].fillna("").astype(str)

cs_human = pd.DataFrame({
    "domain": "CS",
    "label": "Human",
    "title": cs_df[CS_TITLE_COL],
    "text_raw": cs_df[CS_HUMAN_COL],
    "source_column": CS_HUMAN_COL
})

cs_ai = pd.DataFrame({
    "domain": "CS",
    "label": "AI",
    "title": cs_df[CS_TITLE_COL],
    "text_raw": cs_df[CS_AI_COL],
    "source_column": CS_AI_COL
})

med_human = pd.DataFrame({
    "domain": "Medical",
    "label": "Human",
    "title": med_df[MED_TITLE_COL],
    "text_raw": med_df[MED_HUMAN_COL],
    "source_column": MED_HUMAN_COL
})

med_ai = pd.DataFrame({
    "domain": "Medical",
    "label": "AI",
    "title": med_df[MED_TITLE_COL],
    "text_raw": med_df[MED_AI_COL],
    "source_column": MED_AI_COL
})

corpus_df = pd.concat([cs_human, cs_ai, med_human, med_ai], ignore_index=True)
corpus_df["doc_id"] = [f"DOC_{i:05d}" for i in range(len(corpus_df))]
corpus_df = corpus_df[["doc_id", "domain", "label", "title", "source_column", "text_raw"]]

print("========== STEP 1 CORPUS COLLECTION ==========")
print("Long-format corpus shape:", corpus_df.shape)
print()
print(corpus_df.groupby(["domain", "label"]).size())
display(corpus_df.head())

In [ ]:
# ============================================================
# STEP 1 — Corpus Collection
# CS + Medical | AI vs Human
# ============================================================

def read_csv_safely(path):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None

    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"Loaded {os.path.basename(path)} using encoding: {enc}")
            return df
        except Exception as e:
            last_error = e
            print(f"Failed with {enc}: {e}")

    raise RuntimeError(f"Could not read file: {path}. Last error: {last_error}")

cs_df = read_csv_safely(CS_FILE)
med_df = read_csv_safely(MED_FILE)

cs_df = cs_df.drop(columns=[c for c in cs_df.columns if str(c).startswith("Unnamed")], errors="ignore")
med_df = med_df.drop(columns=[c for c in med_df.columns if str(c).startswith("Unnamed")], errors="ignore")

print("CS shape:", cs_df.shape)
print("CS columns:", cs_df.columns.tolist())
print()
print("Medical shape:", med_df.shape)
print("Medical columns:", med_df.columns.tolist())

# ============================================================
# Column names from your original notebook
# Change only if your CSV column names are different
# ============================================================

CS_TITLE_COL = "title"
CS_HUMAN_COL = "original_abstract"
CS_AI_COL = "ai_generated_abstract"

MED_TITLE_COL = "title"
MED_HUMAN_COL = "abstract"
MED_AI_COL = "ai_generated_abstract"

required_cs = [CS_TITLE_COL, CS_HUMAN_COL, CS_AI_COL]
required_med = [MED_TITLE_COL, MED_HUMAN_COL, MED_AI_COL]

for col in required_cs:
    if col not in cs_df.columns:
        raise ValueError(f"CS file missing column: {col}")

for col in required_med:
    if col not in med_df.columns:
        raise ValueError(f"Medical file missing column: {col}")

for col in required_cs:
    cs_df[col] = cs_df[col].fillna("").astype(str)

for col in required_med:
    med_df[col] = med_df[col].fillna("").astype(str)

cs_human = pd.DataFrame({
    "domain": "CS",
    "label": "Human",
    "title": cs_df[CS_TITLE_COL],
    "text_raw": cs_df[CS_HUMAN_COL],
    "source_column": CS_HUMAN_COL
})

cs_ai = pd.DataFrame({
    "domain": "CS",
    "label": "AI",
    "title": cs_df[CS_TITLE_COL],
    "text_raw": cs_df[CS_AI_COL],
    "source_column": CS_AI_COL
})

med_human = pd.DataFrame({
    "domain": "Medical",
    "label": "Human",
    "title": med_df[MED_TITLE_COL],
    "text_raw": med_df[MED_HUMAN_COL],
    "source_column": MED_HUMAN_COL
})

med_ai = pd.DataFrame({
    "domain": "Medical",
    "label": "AI",
    "title": med_df[MED_TITLE_COL],
    "text_raw": med_df[MED_AI_COL],
    "source_column": MED_AI_COL
})

corpus_df = pd.concat([cs_human, cs_ai, med_human, med_ai], ignore_index=True)
corpus_df["doc_id"] = [f"DOC_{i:05d}" for i in range(len(corpus_df))]
corpus_df = corpus_df[["doc_id", "domain", "label", "title", "source_column", "text_raw"]]

print("========== STEP 1 CORPUS COLLECTION ==========")
print("Long-format corpus shape:", corpus_df.shape)
print()
print(corpus_df.groupby(["domain", "label"]).size())
display(corpus_df.head())

In [ ]:
# ============================================================
# FIX CELL BEFORE CELL 5
# This redefines all cleaning functions needed by Cell 5
# ============================================================

!pip install -q ftfy beautifulsoup4

import re
import html
import pandas as pd
from bs4 import BeautifulSoup
import ftfy
from tqdm.auto import tqdm

tqdm.pandas()

def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x)

def normalize_whitespace(text):
    text = safe_str(text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def fix_encoding(text):
    text = safe_str(text)
    text = ftfy.fix_text(text)

    replacements = {
        "Ã¢ÂÂ": "'",
        "Ã¢ÂÂ": "'",
        "Ã¢ÂÂ": '"',
        "Ã¢ÂÂ": '"',
        "Ã¢ÂÂ": "-",
        "Ã¢ÂÂ": "-",
        "Ã¢ÂÂ¦": "...",
        "â€™": "'",
        "â€˜": "'",
        "â€œ": '"',
        "â€": '"',
        "â€“": "-",
        "â€”": "-",
        "â€¦": "...",
        "Â": "",
        "\x81": "",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    return text

def remove_html(text):
    text = safe_str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]*>", " ", text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    return text

def remove_prompt_echo(text):
    text = safe_str(text)
    text = normalize_whitespace(text)

    leading_patterns = [
        r"^\s*here is .*?abstract[:\-]?\s*",
        r"^\s*here are .*?abstracts[:\-]?\s*",
        r"^\s*below is .*?abstract[:\-]?\s*",
        r"^\s*the following is .*?abstract[:\-]?\s*",
        r"^\s*this is .*?abstract[:\-]?\s*",
        r"^\s*rewritten version[:\-]?\s*",
        r"^\s*rewritten abstract[:\-]?\s*",
        r"^\s*new ai generated abstract[:\-]?\s*",
        r"^\s*ai generated abstract[:\-]?\s*",
        r"^\s*generated abstract[:\-]?\s*",
        r"^\s*certainly[,.]?\s*",
        r"^\s*sure[,.]?\s*",
        r"^\s*of course[,.]?\s*",
    ]

    for pat in leading_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    label_patterns = [
        r"\bNew AI Generated Abstract\s*:\s*",
        r"\bAI Generated Abstract\s*:\s*",
        r"\bGenerated Abstract\s*:\s*",
        r"\bRewritten Abstract\s*:\s*",
        r"\bOriginal Abstract\s*:\s*",
        r"\bHuman Abstract\s*:\s*",
        r"\bTitle\s*:\s*",
        r"\bAbstract\s*:\s*",
    ]

    for pat in label_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    phrase_patterns = [
        r"\bhere is a rewritten version\b",
        r"\bhere is the rewritten version\b",
        r"\bhere is the abstract\b",
        r"\bhere is an abstract\b",
        r"\bthe rewritten abstract is\b",
        r"\bthis rewritten abstract\b",
    ]

    for pat in phrase_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    return normalize_whitespace(text)

def clean_text_basic(text, label):
    text = safe_str(text)
    text = fix_encoding(text)
    text = remove_html(text)

    if label == "AI":
        text = remove_prompt_echo(text)

    text = normalize_whitespace(text)
    return text

def word_count_raw(text):
    text = safe_str(text)
    return len(re.findall(r"\b\w+\b", text))

def contains_html(text):
    return bool(re.search(r"<[^>]+>", safe_str(text)))

def contains_encoding_artifact(text):
    bad_patterns = ["Ã", "Â", "â€", "Ã¢", "\x81"]
    text = safe_str(text)
    return any(p in text for p in bad_patterns)

def contains_prompt_echo(text):
    low = safe_str(text).lower()
    patterns = [
        "here is the abstract",
        "here is a rewritten",
        "rewritten abstract:",
        "new ai generated abstract:",
        "ai generated abstract:",
        "original abstract:",
        "generated abstract:",
    ]
    return any(p in low for p in patterns)

# quick test
print(clean_text_basic("Here is the abstract: <b>This is a test.</b>", "AI"))
print("clean_text_basic is now defined successfully.")

In [ ]:
corpus_df["text_clean"] = corpus_df.progress_apply(
    lambda row: clean_text_basic(row["text_raw"], row["label"]),
    axis=1
)

corpus_df["raw_word_count"] = corpus_df["text_raw"].apply(word_count_raw)
corpus_df["clean_word_count"] = corpus_df["text_clean"].apply(word_count_raw)

quality_counts = {
    "total_docs": len(corpus_df),
    "empty_text_clean": int((corpus_df["text_clean"].str.strip() == "").sum()),
    "html_remaining": int(corpus_df["text_clean"].apply(contains_html).sum()),
    "encoding_artifact_remaining": int(corpus_df["text_clean"].apply(contains_encoding_artifact).sum()),
    "prompt_echo_remaining_in_AI": int(
        corpus_df.loc[corpus_df["label"] == "AI", "text_clean"].apply(contains_prompt_echo).sum()
    ),
    "human_under_50_words": int(
        ((corpus_df["label"] == "Human") & (corpus_df["clean_word_count"] < 50)).sum()
    ),
    "ai_under_100_words": int(
        ((corpus_df["label"] == "AI") & (corpus_df["clean_word_count"] < 100)).sum()
    ),
}

audit_before_after = corpus_df.groupby(["domain", "label"]).agg(
    docs=("doc_id", "count"),
    raw_mean_words=("raw_word_count", "mean"),
    clean_mean_words=("clean_word_count", "mean"),
    raw_min_words=("raw_word_count", "min"),
    clean_min_words=("clean_word_count", "min"),
    raw_max_words=("raw_word_count", "max"),
    clean_max_words=("clean_word_count", "max"),
).reset_index()

print("========== STEP 2 CLEANING AUDIT ==========")
display(audit_before_after)

print("Quality counts:")
for k, v in quality_counts.items():
    print(f"{k}: {v}")

display(corpus_df[["doc_id", "domain", "label", "title", "clean_word_count", "text_clean"]].head())

In [ ]:
import nltk
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))

# Keep useful contrast words that may matter in writing style
keep_words = {
    "not", "no", "nor",
    "against",
    "between",
    "under",
    "over",
    "more",
    "most"
}
stop_words = stop_words - keep_words

lemmatizer = WordNetLemmatizer()

def preprocess_for_tfidf(text):
    text = safe_str(text).lower()

    tokens = re.findall(r"\b[a-zA-Z]+\b", text)

    processed = []

    for tok in tokens:
        tok = tok.lower().strip()

        if len(tok) < 2:
            continue

        if tok in stop_words:
            continue

        lemma = lemmatizer.lemmatize(tok)

        if len(lemma) < 2:
            continue

        processed.append(lemma)

    return processed

corpus_df["tokens"] = corpus_df["text_clean"].progress_apply(preprocess_for_tfidf)
corpus_df["preprocessed_text"] = corpus_df["tokens"].apply(lambda toks: " ".join(toks))
corpus_df["token_count"] = corpus_df["tokens"].apply(len)

token_audit = corpus_df.groupby(["domain", "label"]).agg(
    docs=("doc_id", "count"),
    mean_tokens=("token_count", "mean"),
    min_tokens=("token_count", "min"),
    max_tokens=("token_count", "max"),
).reset_index()

print("========== STEP 2 TOKENIZATION + LEMMATIZATION AUDIT ==========")
display(token_audit)

print("Empty preprocessed documents:", int((corpus_df["preprocessed_text"].str.strip() == "").sum()))

STEP2_LONG_OUTPUT = os.path.join(STEP2_OUTPUT_DIR, "step2_long_corpus_cleaned.csv")
corpus_df.to_csv(STEP2_LONG_OUTPUT, index=False, encoding="utf-8-sig")

print("Saved:", STEP2_LONG_OUTPUT)

In [ ]:
from scipy import sparse

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

import umap.umap_ as umap

def label_to_int(labels):
    return np.array([1 if x == "AI" else 0 for x in labels])

def purity_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    total = 0
    for cluster in np.unique(y_pred):
        idx = y_pred == cluster
        _, counts = np.unique(y_true[idx], return_counts=True)
        total += counts.max()

    return total / len(y_true)

def evaluate_clustering(X, true_labels, pred_labels, domain_name, representation_name, algorithm_name):
    y_true_int = label_to_int(true_labels)

    purity = purity_score(y_true_int, pred_labels)
    ari = adjusted_rand_score(y_true_int, pred_labels)
    nmi = normalized_mutual_info_score(y_true_int, pred_labels)

    try:
        sil = silhouette_score(X, pred_labels)
    except Exception:
        sil = np.nan

    return {
        "representation": representation_name,
        "domain": domain_name,
        "algorithm": algorithm_name,
        "documents": len(true_labels),
        "human_docs": int((np.asarray(true_labels) == "Human").sum()),
        "ai_docs": int((np.asarray(true_labels) == "AI").sum()),
        "purity": purity,
        "ari": ari,
        "nmi": nmi,
        "silhouette": sil,
    }

def cluster_composition(true_labels, pred_labels):
    comp = pd.crosstab(
        pd.Series(pred_labels, name="cluster"),
        pd.Series(true_labels, name="true_label")
    )
    return comp

def run_clustering_suite(X, true_labels, domain_name, representation_name):
    rows = []
    predictions = {}

    # 1. KMeans
    kmeans = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=30)
    pred_kmeans = kmeans.fit_predict(X)
    predictions["KMeans"] = pred_kmeans
    rows.append(evaluate_clustering(X, true_labels, pred_kmeans, domain_name, representation_name, "KMeans"))

    # 2. Agglomerative clustering
    agg = AgglomerativeClustering(n_clusters=2, linkage="ward")
    pred_agg = agg.fit_predict(X)
    predictions["Agglomerative"] = pred_agg
    rows.append(evaluate_clustering(X, true_labels, pred_agg, domain_name, representation_name, "Agglomerative"))

    # 3. Gaussian Mixture Model
    gmm = GaussianMixture(n_components=2, random_state=RANDOM_STATE, covariance_type="full")
    pred_gmm = gmm.fit_predict(X)
    predictions["GMM"] = pred_gmm
    rows.append(evaluate_clustering(X, true_labels, pred_gmm, domain_name, representation_name, "GMM"))

    metrics_df = pd.DataFrame(rows)

    print(f"========== {representation_name} | {domain_name} CLUSTERING RESULTS ==========")
    display(metrics_df)

    for algo, pred in predictions.items():
        print(f"\n--- {algo} cluster composition ---")
        display(cluster_composition(true_labels, pred))

    return metrics_df, predictions

def fit_umap_source_then_transfer(X_source, X_target, n_components=10, metric="cosine"):
    reducer = umap.UMAP(
        n_components=n_components,
        n_neighbors=15,
        min_dist=0.05,
        metric=metric,
        random_state=RANDOM_STATE
    )

    Z_source = reducer.fit_transform(X_source)
    Z_target = reducer.transform(X_target)

    return Z_source, Z_target, reducer

def plot_umap_2d(Z, labels, title):
    plt.figure(figsize=(7, 5))

    labels = np.asarray(labels)
    for lab in np.unique(labels):
        idx = labels == lab
        plt.scatter(Z[idx, 0], Z[idx, 1], s=8, alpha=0.6, label=str(lab))

    plt.title(title)
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.legend()
    plt.show()

def jaccard_score_sets(a, b):
    a = set(a)
    b = set(b)
    if len(a.union(b)) == 0:
        return np.nan
    return len(a.intersection(b)) / len(a.union(b))

In [ ]:
# ============================================================
# STEP 3 — TF-IDF fitted on Domain A / CS only
# ============================================================

cs_docs = corpus_df[
    (corpus_df["domain"] == "CS") &
    (corpus_df["preprocessed_text"].str.strip() != "")
].copy().reset_index(drop=True)

med_docs = corpus_df[
    (corpus_df["domain"] == "Medical") &
    (corpus_df["preprocessed_text"].str.strip() != "")
].copy().reset_index(drop=True)

cs_docs["cs_row_id"] = np.arange(len(cs_docs))
med_docs["med_row_id"] = np.arange(len(med_docs))

print("========== DOMAIN A SELECTION ==========")
print("CS documents:", len(cs_docs))
print(cs_docs["label"].value_counts())
print()
print("Medical documents used in Step 3:", int((cs_docs["domain"] == "Medical").sum()))

TFIDF_CONFIG = {
    "ngram_range": (1, 2),
    "min_df": 5,
    "max_df": 0.85,
    "max_features": 30000,
    "sublinear_tf": True,
    "norm": "l2",
    "lowercase": False,
    "dtype": np.float32
}

vectorizer_cs = TfidfVectorizer(**TFIDF_CONFIG)
X_cs_tfidf = vectorizer_cs.fit_transform(cs_docs["preprocessed_text"])

feature_names = np.array(vectorizer_cs.get_feature_names_out())

features_df = pd.DataFrame({
    "feature_id": np.arange(len(feature_names)),
    "feature": feature_names,
    "idf": vectorizer_cs.idf_
})

features_df["ngram_type"] = features_df["feature"].apply(
    lambda x: "bigram" if " " in x else "unigram"
)

print("========== STEP 3 TF-IDF FIT COMPLETE ==========")
print("TF-IDF matrix shape:", X_cs_tfidf.shape)
print("Number of features:", len(feature_names))
print("Non-zero values:", X_cs_tfidf.nnz)
print("Sparsity:", 1 - (X_cs_tfidf.nnz / (X_cs_tfidf.shape[0] * X_cs_tfidf.shape[1])))
print("All-zero rows:", int((np.diff(X_cs_tfidf.indptr) == 0).sum()))

display(features_df.head())

joblib.dump(vectorizer_cs, os.path.join(TFIDF_OUTPUT_DIR, "tfidf_vectorizer_cs_only.joblib"))
features_df.to_csv(os.path.join(TFIDF_OUTPUT_DIR, "tfidf_features_cs_only.csv"), index=False)

In [ ]:
# ============================================================
# STEP 4 — Contrast Computation
# TFIDF_AI - TFIDF_Human
# ============================================================

ai_mask_cs = cs_docs["label"].values == "AI"
human_mask_cs = cs_docs["label"].values == "Human"

X_cs_ai = X_cs_tfidf[ai_mask_cs]
X_cs_human = X_cs_tfidf[human_mask_cs]

mean_tfidf_ai = np.asarray(X_cs_ai.mean(axis=0)).ravel()
mean_tfidf_human = np.asarray(X_cs_human.mean(axis=0)).ravel()

contrast_ai_minus_human = mean_tfidf_ai - mean_tfidf_human
abs_contrast = np.abs(contrast_ai_minus_human)

tfidf_contrast_df = pd.DataFrame({
    "feature_id": np.arange(len(feature_names)),
    "feature": feature_names,
    "mean_tfidf_ai": mean_tfidf_ai,
    "mean_tfidf_human": mean_tfidf_human,
    "contrast_ai_minus_human": contrast_ai_minus_human,
    "abs_contrast": abs_contrast,
    "idf": vectorizer_cs.idf_,
})

tfidf_contrast_df["direction"] = np.where(
    tfidf_contrast_df["contrast_ai_minus_human"] > 0,
    "AI-dominant",
    np.where(tfidf_contrast_df["contrast_ai_minus_human"] < 0, "Human-dominant", "Tie")
)

print("========== STEP 4 TF-IDF CONTRAST ==========")
print("Total features:", len(tfidf_contrast_df))
print("AI-dominant features:", int((tfidf_contrast_df["contrast_ai_minus_human"] > 0).sum()))
print("Human-dominant features:", int((tfidf_contrast_df["contrast_ai_minus_human"] < 0).sum()))
print("Tie features:", int((tfidf_contrast_df["contrast_ai_minus_human"] == 0).sum()))
print("Min contrast:", tfidf_contrast_df["contrast_ai_minus_human"].min())
print("Max contrast:", tfidf_contrast_df["contrast_ai_minus_human"].max())

display(tfidf_contrast_df.sort_values("contrast_ai_minus_human", ascending=False).head(10))
display(tfidf_contrast_df.sort_values("contrast_ai_minus_human", ascending=True).head(10))

tfidf_contrast_df.to_csv(
    os.path.join(TFIDF_OUTPUT_DIR, "step4_tfidf_all_contrast_scores.csv"),
    index=False
)

In [ ]:
# ============================================================
# STEP 5 — Contrast Vocabulary
# Top AI words + Top Human words
# ============================================================

def is_artifact_feature(term):
    term = safe_str(term).lower().strip()

    artifact_patterns = [
        r"http",
        r"www",
        r"github",
        r"\.com",
        r"\.org",
        r"\.net",
        r"\\",
        r"\$",
        r"[_{}<>]",
        r"\bdoi\b",
        r"\barxiv\b",
        r"\bhtml\b",
        r"\bxml\b",
        r"\bpdf\b",
        r"\burl\b",
        r"\bfig\b",
        r"\bfigure\b",
        r"\btable\b",
    ]

    if len(term) < 2:
        return True

    if re.search(r"\d", term):
        return True

    return any(re.search(p, term) for p in artifact_patterns)

tfidf_contrast_clean = tfidf_contrast_df[
    ~tfidf_contrast_df["feature"].apply(is_artifact_feature)
].copy()

TOP_N_AI = 100
TOP_N_HUMAN = 100

top_ai_tfidf = (
    tfidf_contrast_clean[tfidf_contrast_clean["contrast_ai_minus_human"] > 0]
    .sort_values("contrast_ai_minus_human", ascending=False)
    .head(TOP_N_AI)
    .copy()
)

top_human_tfidf = (
    tfidf_contrast_clean[tfidf_contrast_clean["contrast_ai_minus_human"] < 0]
    .sort_values("contrast_ai_minus_human", ascending=True)
    .head(TOP_N_HUMAN)
    .copy()
)

top_ai_tfidf["vocab_group"] = "AI"
top_human_tfidf["vocab_group"] = "Human"

selected_tfidf_vocab = pd.concat([top_ai_tfidf, top_human_tfidf], ignore_index=True)

selected_tfidf_feature_ids = selected_tfidf_vocab["feature_id"].values
selected_tfidf_terms = selected_tfidf_vocab["feature"].values

print("========== STEP 5 TF-IDF CONTRAST VOCABULARY ==========")
print("Original features:", len(tfidf_contrast_df))
print("After artifact filtering:", len(tfidf_contrast_clean))
print("Selected AI terms:", len(top_ai_tfidf))
print("Selected Human terms:", len(top_human_tfidf))
print("Total selected vocabulary size:", len(selected_tfidf_vocab))
print("Duplicate selected features:", selected_tfidf_vocab["feature"].duplicated().sum())
print("AI/Human overlap:", len(set(top_ai_tfidf["feature"]).intersection(set(top_human_tfidf["feature"]))))

print("\nTop AI terms:")
display(top_ai_tfidf[["feature", "contrast_ai_minus_human"]].head(20))

print("\nTop Human terms:")
display(top_human_tfidf[["feature", "contrast_ai_minus_human"]].head(20))

selected_tfidf_vocab.to_csv(
    os.path.join(TFIDF_OUTPUT_DIR, "step5_selected_tfidf_contrast_vocabulary.csv"),
    index=False
)

In [ ]:
# ============================================================
# STEP 6 — DTM Construction for Domain A / CS
# Select the 200 contrast-vocabulary TF-IDF columns
# ============================================================

X_cs_tfidf_dtm = X_cs_tfidf[:, selected_tfidf_feature_ids]

print("========== STEP 6 TF-IDF DTM DOMAIN A / CS ==========")
print("DTM shape:", X_cs_tfidf_dtm.shape)
print("Rows/documents:", X_cs_tfidf_dtm.shape[0])
print("Vocabulary terms:", X_cs_tfidf_dtm.shape[1])
print("Non-zero values:", X_cs_tfidf_dtm.nnz)
print("Sparsity:", 1 - (X_cs_tfidf_dtm.nnz / (X_cs_tfidf_dtm.shape[0] * X_cs_tfidf_dtm.shape[1])))
print("All-zero rows:", int((np.diff(X_cs_tfidf_dtm.indptr) == 0).sum()))

print("AI avg non-zero terms/doc:", np.diff(X_cs_tfidf_dtm[ai_mask_cs].indptr).mean())
print("Human avg non-zero terms/doc:", np.diff(X_cs_tfidf_dtm[human_mask_cs].indptr).mean())

In [ ]:
# ============================================================
# STEP 7 — UMAP + Clustering on Domain A / CS
# UMAP is applied before clustering
# ============================================================

X_cs_tfidf_dtm_norm = normalize(X_cs_tfidf_dtm, norm="l2", axis=1)
X_cs_tfidf_dtm_dense = X_cs_tfidf_dtm_norm.toarray().astype(np.float32)

# Temporary placeholder for Medical, actual Medical transform is done after Step 8/9.
# Here we fit UMAP on CS only.
tfidf_umap_reducer = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.05,
    metric="cosine",
    random_state=RANDOM_STATE
)

Z_cs_tfidf_umap = tfidf_umap_reducer.fit_transform(X_cs_tfidf_dtm_dense)

print("========== STEP 7 TF-IDF + UMAP DOMAIN A ==========")
print("Original CS DTM shape:", X_cs_tfidf_dtm_dense.shape)
print("UMAP-reduced CS shape:", Z_cs_tfidf_umap.shape)

tfidf_cs_metrics, tfidf_cs_preds = run_clustering_suite(
    Z_cs_tfidf_umap,
    cs_docs["label"].values,
    domain_name="Domain A / CS",
    representation_name="TF-IDF + UMAP"
)

plot_umap_2d(
    Z_cs_tfidf_umap,
    cs_docs["label"].values,
    "TF-IDF + UMAP | Domain A / CS by true label"
)

plot_umap_2d(
    Z_cs_tfidf_umap,
    tfidf_cs_preds["KMeans"],
    "TF-IDF + UMAP | Domain A / CS by KMeans cluster"
)

In [ ]:
# ============================================================
# STEP 8 — Apply SAME CS-fitted TF-IDF vocabulary to Medical
# Use transform(), not fit_transform()
# ============================================================

X_med_tfidf_full = vectorizer_cs.transform(med_docs["preprocessed_text"])

print("========== STEP 8 TF-IDF CROSS-DOMAIN TRANSFER ==========")
print("Medical documents:", len(med_docs))
print(med_docs["label"].value_counts())
print("Full Medical TF-IDF shape:", X_med_tfidf_full.shape)
print("Selected vocabulary terms:", len(selected_tfidf_feature_ids))
print("Feature mismatch:", X_med_tfidf_full.shape[1] != X_cs_tfidf.shape[1])

In [ ]:
# ============================================================
# STEP 9 — DTM Construction for Domain B / Medical
# Select the same 200 CS-learned contrast vocabulary columns
# ============================================================

X_med_tfidf_dtm = X_med_tfidf_full[:, selected_tfidf_feature_ids]

ai_mask_med = med_docs["label"].values == "AI"
human_mask_med = med_docs["label"].values == "Human"

print("========== STEP 9 TF-IDF DTM DOMAIN B / MEDICAL ==========")
print("DTM shape:", X_med_tfidf_dtm.shape)
print("Rows/documents:", X_med_tfidf_dtm.shape[0])
print("Vocabulary terms:", X_med_tfidf_dtm.shape[1])
print("Non-zero values:", X_med_tfidf_dtm.nnz)
print("Sparsity:", 1 - (X_med_tfidf_dtm.nnz / (X_med_tfidf_dtm.shape[0] * X_med_tfidf_dtm.shape[1])))
print("All-zero rows:", int((np.diff(X_med_tfidf_dtm.indptr) == 0).sum()))

print("AI avg non-zero terms/doc:", np.diff(X_med_tfidf_dtm[ai_mask_med].indptr).mean())
print("Human avg non-zero terms/doc:", np.diff(X_med_tfidf_dtm[human_mask_med].indptr).mean())

print("AI all-zero rows:", int((np.diff(X_med_tfidf_dtm[ai_mask_med].indptr) == 0).sum()))
print("Human all-zero rows:", int((np.diff(X_med_tfidf_dtm[human_mask_med].indptr) == 0).sum()))

In [ ]:
# ============================================================
# STEP 10 — UMAP + Clustering on Domain B / Medical
# Use same UMAP reducer fitted on CS
# ============================================================

X_med_tfidf_dtm_norm = normalize(X_med_tfidf_dtm, norm="l2", axis=1)
X_med_tfidf_dtm_dense = X_med_tfidf_dtm_norm.toarray().astype(np.float32)

Z_med_tfidf_umap = tfidf_umap_reducer.transform(X_med_tfidf_dtm_dense)

print("========== STEP 10 TF-IDF + UMAP DOMAIN B ==========")
print("Original Medical DTM shape:", X_med_tfidf_dtm_dense.shape)
print("UMAP-reduced Medical shape:", Z_med_tfidf_umap.shape)

tfidf_med_metrics, tfidf_med_preds = run_clustering_suite(
    Z_med_tfidf_umap,
    med_docs["label"].values,
    domain_name="Domain B / Medical",
    representation_name="TF-IDF + UMAP"
)

plot_umap_2d(
    Z_med_tfidf_umap,
    med_docs["label"].values,
    "TF-IDF + UMAP | Domain B / Medical by true label"
)

plot_umap_2d(
    Z_med_tfidf_umap,
    tfidf_med_preds["KMeans"],
    "TF-IDF + UMAP | Domain B / Medical by KMeans cluster"
)

In [ ]:
# ============================================================
# STEP 11 — TF-IDF Evaluation
# Purity | ARI | NMI | Coherence | Jaccard | UMAP
# ============================================================

# ---------- Medical contrast using same selected vocabulary ----------
X_med_ai_selected = X_med_tfidf_dtm[ai_mask_med]
X_med_human_selected = X_med_tfidf_dtm[human_mask_med]

mean_med_ai_selected = np.asarray(X_med_ai_selected.mean(axis=0)).ravel()
mean_med_human_selected = np.asarray(X_med_human_selected.mean(axis=0)).ravel()

med_contrast_selected = mean_med_ai_selected - mean_med_human_selected

tfidf_stability_df = selected_tfidf_vocab.copy()
tfidf_stability_df["medical_mean_tfidf_ai"] = mean_med_ai_selected
tfidf_stability_df["medical_mean_tfidf_human"] = mean_med_human_selected
tfidf_stability_df["medical_contrast_ai_minus_human"] = med_contrast_selected

tfidf_stability_df["cs_sign"] = np.sign(tfidf_stability_df["contrast_ai_minus_human"])
tfidf_stability_df["medical_sign"] = np.sign(tfidf_stability_df["medical_contrast_ai_minus_human"])
tfidf_stability_df["same_direction"] = tfidf_stability_df["cs_sign"] == tfidf_stability_df["medical_sign"]

# ---------- Presence Jaccard ----------
med_presence_mask = np.asarray((X_med_tfidf_dtm > 0).sum(axis=0)).ravel() > 0
cs_selected_terms = set(selected_tfidf_vocab["feature"])
med_present_terms = set(selected_tfidf_vocab.loc[med_presence_mask, "feature"])

presence_jaccard_all = jaccard_score_sets(cs_selected_terms, med_present_terms)

cs_ai_terms = set(top_ai_tfidf["feature"])
cs_human_terms = set(top_human_tfidf["feature"])

med_ai_direction_terms = set(
    tfidf_stability_df.loc[tfidf_stability_df["medical_contrast_ai_minus_human"] > 0, "feature"]
)
med_human_direction_terms = set(
    tfidf_stability_df.loc[tfidf_stability_df["medical_contrast_ai_minus_human"] < 0, "feature"]
)

directional_jaccard_ai = jaccard_score_sets(cs_ai_terms, med_ai_direction_terms)
directional_jaccard_human = jaccard_score_sets(cs_human_terms, med_human_direction_terms)

overall_direction_agreement = tfidf_stability_df["same_direction"].mean()
ai_direction_agreement = tfidf_stability_df.loc[
    tfidf_stability_df["vocab_group"] == "AI", "same_direction"
].mean()
human_direction_agreement = tfidf_stability_df.loc[
    tfidf_stability_df["vocab_group"] == "Human", "same_direction"
].mean()

tfidf_stability_summary = pd.DataFrame([{
    "representation": "TF-IDF + UMAP",
    "presence_jaccard_all_terms": presence_jaccard_all,
    "directional_jaccard_ai_terms": directional_jaccard_ai,
    "directional_jaccard_human_terms": directional_jaccard_human,
    "overall_direction_agreement": overall_direction_agreement,
    "ai_vocab_direction_agreement": ai_direction_agreement,
    "human_vocab_direction_agreement": human_direction_agreement,
}])

print("========== TF-IDF STABILITY SUMMARY ==========")
display(tfidf_stability_summary)

# ---------- NPMI coherence ----------
def npmi_coherence_from_matrix(X, col_positions, eps=1e-12):
    """
    X: document-term matrix using selected vocabulary
    col_positions: positions within selected vocabulary
    """
    if sparse.issparse(X):
        B = (X[:, col_positions] > 0).astype(np.int32).toarray()
    else:
        B = (X[:, col_positions] > 0).astype(np.int32)

    n_docs = B.shape[0]
    n_terms = B.shape[1]

    if n_terms < 2:
        return {
            "mean_npmi": np.nan,
            "median_npmi": np.nan,
            "zero_cooccurrence_pairs": np.nan
        }

    df = B.sum(axis=0)
    co = B.T @ B

    scores = []
    zero_pairs = 0

    for i in range(n_terms):
        for j in range(i + 1, n_terms):
            p_i = df[i] / n_docs
            p_j = df[j] / n_docs
            p_ij = co[i, j] / n_docs

            if p_ij <= 0:
                zero_pairs += 1
                continue

            pmi = np.log((p_ij + eps) / ((p_i * p_j) + eps))
            npmi = pmi / (-np.log(p_ij + eps))
            scores.append(npmi)

    if len(scores) == 0:
        return {
            "mean_npmi": np.nan,
            "median_npmi": np.nan,
            "zero_cooccurrence_pairs": zero_pairs
        }

    return {
        "mean_npmi": float(np.mean(scores)),
        "median_npmi": float(np.median(scores)),
        "zero_cooccurrence_pairs": int(zero_pairs)
    }

ai_positions = list(range(0, TOP_N_AI))
human_positions = list(range(TOP_N_AI, TOP_N_AI + TOP_N_HUMAN))
all_positions = list(range(TOP_N_AI + TOP_N_HUMAN))

coherence_rows = []

for domain_name, Xmat in [
    ("CS", X_cs_tfidf_dtm),
    ("Medical", X_med_tfidf_dtm)
]:
    for vocab_name, positions in [
        ("AI contrast vocab", ai_positions),
        ("Human contrast vocab", human_positions),
        ("All contrast vocab", all_positions)
    ]:
        result = npmi_coherence_from_matrix(Xmat, positions)
        coherence_rows.append({
            "representation": "TF-IDF",
            "domain": domain_name,
            "vocabulary_group": vocab_name,
            **result
        })

tfidf_coherence_df = pd.DataFrame(coherence_rows)

print("========== TF-IDF VOCABULARY COHERENCE ==========")
display(tfidf_coherence_df)

tfidf_stability_df.to_csv(
    os.path.join(TFIDF_OUTPUT_DIR, "step11_tfidf_directional_stability.csv"),
    index=False
)
tfidf_stability_summary.to_csv(
    os.path.join(TFIDF_OUTPUT_DIR, "step11_tfidf_stability_summary.csv"),
    index=False
)
tfidf_coherence_df.to_csv(
    os.path.join(TFIDF_OUTPUT_DIR, "step11_tfidf_coherence_npmi.csv"),
    index=False
)

PART B — SciBERT + UMAP pipeline

In [ ]:
# ============================================================
# STEP 3 — SciBERT representation on Domain A / CS only
# Instead of fitting TF-IDF vocabulary, extract SciBERT embeddings
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModel

SCIBERT_MODEL_NAME = "allenai/scibert_scivocab_uncased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(SCIBERT_MODEL_NAME)
scibert_model = AutoModel.from_pretrained(SCIBERT_MODEL_NAME).to(device)
scibert_model.eval()

CS_SCIBERT_FILE = os.path.join(SCIBERT_OUTPUT_DIR, "step3_cs_scibert_embeddings.npy")
MED_SCIBERT_FILE = os.path.join(SCIBERT_OUTPUT_DIR, "step8_medical_scibert_embeddings.npy")

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def embed_texts_scibert(texts, batch_size=16, max_length=512):
    all_embeddings = []

    texts = list(texts)

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = scibert_model(**encoded)
            embeddings = mean_pooling(outputs.last_hidden_state, encoded["attention_mask"])

        all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings).astype(np.float32)

if os.path.exists(CS_SCIBERT_FILE):
    E_cs_scibert_raw = np.load(CS_SCIBERT_FILE)
    print("Loaded existing CS SciBERT embeddings:", E_cs_scibert_raw.shape)
else:
    # For SciBERT, use cleaned natural text, not lemmatized TF-IDF text
    E_cs_scibert_raw = embed_texts_scibert(
        cs_docs["text_clean"].fillna("").astype(str).values,
        batch_size=16,
        max_length=512
    )
    np.save(CS_SCIBERT_FILE, E_cs_scibert_raw)
    print("Saved CS SciBERT embeddings:", CS_SCIBERT_FILE)

print("========== STEP 3 SCIBERT DOMAIN A ==========")
print("CS SciBERT embedding shape:", E_cs_scibert_raw.shape)

In [ ]:
# ============================================================
# STEP 4 — SciBERT Contrast Computation
# EMBEDDING_AI - EMBEDDING_HUMAN
# ============================================================

# Fit scaler on CS only
scibert_scaler = StandardScaler()
E_cs_scibert = scibert_scaler.fit_transform(E_cs_scibert_raw)

E_cs_ai = E_cs_scibert[ai_mask_cs]
E_cs_human = E_cs_scibert[human_mask_cs]

mean_emb_ai = E_cs_ai.mean(axis=0)
mean_emb_human = E_cs_human.mean(axis=0)

scibert_contrast = mean_emb_ai - mean_emb_human
scibert_abs_contrast = np.abs(scibert_contrast)

scibert_contrast_df = pd.DataFrame({
    "dimension_id": np.arange(E_cs_scibert.shape[1]),
    "mean_scibert_ai": mean_emb_ai,
    "mean_scibert_human": mean_emb_human,
    "contrast_ai_minus_human": scibert_contrast,
    "abs_contrast": scibert_abs_contrast,
})

scibert_contrast_df["direction"] = np.where(
    scibert_contrast_df["contrast_ai_minus_human"] > 0,
    "AI-dominant",
    np.where(scibert_contrast_df["contrast_ai_minus_human"] < 0, "Human-dominant", "Tie")
)

print("========== STEP 4 SCIBERT CONTRAST ==========")
print("Total embedding dimensions:", len(scibert_contrast_df))
print("AI-dominant dimensions:", int((scibert_contrast_df["contrast_ai_minus_human"] > 0).sum()))
print("Human-dominant dimensions:", int((scibert_contrast_df["contrast_ai_minus_human"] < 0).sum()))
print("Tie dimensions:", int((scibert_contrast_df["contrast_ai_minus_human"] == 0).sum()))
print("Min contrast:", scibert_contrast_df["contrast_ai_minus_human"].min())
print("Max contrast:", scibert_contrast_df["contrast_ai_minus_human"].max())

display(scibert_contrast_df.sort_values("contrast_ai_minus_human", ascending=False).head(10))
display(scibert_contrast_df.sort_values("contrast_ai_minus_human", ascending=True).head(10))

scibert_contrast_df.to_csv(
    os.path.join(SCIBERT_OUTPUT_DIR, "step4_scibert_dimension_contrast_scores.csv"),
    index=False
)

joblib.dump(scibert_scaler, os.path.join(SCIBERT_OUTPUT_DIR, "scibert_standard_scaler_cs_only.joblib"))

In [ ]:
# ============================================================
# STEP 5 — SciBERT Contrast Feature Selection
# Top AI embedding dimensions + Top Human embedding dimensions
# ============================================================

TOP_N_AI_DIMS = 100
TOP_N_HUMAN_DIMS = 100

top_ai_scibert_dims = (
    scibert_contrast_df[scibert_contrast_df["contrast_ai_minus_human"] > 0]
    .sort_values("contrast_ai_minus_human", ascending=False)
    .head(TOP_N_AI_DIMS)
    .copy()
)

top_human_scibert_dims = (
    scibert_contrast_df[scibert_contrast_df["contrast_ai_minus_human"] < 0]
    .sort_values("contrast_ai_minus_human", ascending=True)
    .head(TOP_N_HUMAN_DIMS)
    .copy()
)

top_ai_scibert_dims["feature_group"] = "AI"
top_human_scibert_dims["feature_group"] = "Human"

selected_scibert_features = pd.concat(
    [top_ai_scibert_dims, top_human_scibert_dims],
    ignore_index=True
)

selected_scibert_dim_ids = selected_scibert_features["dimension_id"].values

print("========== STEP 5 SCIBERT CONTRAST FEATURES ==========")
print("Selected AI-dominant dimensions:", len(top_ai_scibert_dims))
print("Selected Human-dominant dimensions:", len(top_human_scibert_dims))
print("Total selected SciBERT dimensions:", len(selected_scibert_features))
print("Duplicate dimensions:", selected_scibert_features["dimension_id"].duplicated().sum())
print("AI/Human overlap:", len(set(top_ai_scibert_dims["dimension_id"]).intersection(set(top_human_scibert_dims["dimension_id"]))))

print("\nTop AI SciBERT dimensions:")
display(top_ai_scibert_dims[["dimension_id", "contrast_ai_minus_human"]].head(20))

print("\nTop Human SciBERT dimensions:")
display(top_human_scibert_dims[["dimension_id", "contrast_ai_minus_human"]].head(20))

selected_scibert_features.to_csv(
    os.path.join(SCIBERT_OUTPUT_DIR, "step5_selected_scibert_contrast_dimensions.csv"),
    index=False
)

In [ ]:
# ============================================================
# STEP 6 — SciBERT Feature Matrix for Domain A / CS
# Use selected contrast dimensions only
# ============================================================

X_cs_scibert_selected = E_cs_scibert[:, selected_scibert_dim_ids]

# Normalize before UMAP
X_cs_scibert_selected_norm = normalize(X_cs_scibert_selected, norm="l2", axis=1)

print("========== STEP 6 SCIBERT FEATURE MATRIX DOMAIN A / CS ==========")
print("Selected SciBERT matrix shape:", X_cs_scibert_selected_norm.shape)
print("Rows/documents:", X_cs_scibert_selected_norm.shape[0])
print("Selected dimensions:", X_cs_scibert_selected_norm.shape[1])

In [ ]:
# ============================================================
# STEP 7 — SciBERT + UMAP + Clustering on Domain A / CS
# ============================================================

scibert_umap_reducer = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.05,
    metric="cosine",
    random_state=RANDOM_STATE
)

Z_cs_scibert_umap = scibert_umap_reducer.fit_transform(X_cs_scibert_selected_norm)

print("========== STEP 7 SCIBERT + UMAP DOMAIN A ==========")
print("Original selected SciBERT shape:", X_cs_scibert_selected_norm.shape)
print("UMAP-reduced CS shape:", Z_cs_scibert_umap.shape)

scibert_cs_metrics, scibert_cs_preds = run_clustering_suite(
    Z_cs_scibert_umap,
    cs_docs["label"].values,
    domain_name="Domain A / CS",
    representation_name="SciBERT + UMAP"
)

plot_umap_2d(
    Z_cs_scibert_umap,
    cs_docs["label"].values,
    "SciBERT + UMAP | Domain A / CS by true label"
)

plot_umap_2d(
    Z_cs_scibert_umap,
    scibert_cs_preds["KMeans"],
    "SciBERT + UMAP | Domain A / CS by KMeans cluster"
)

In [ ]:
# ============================================================
# STEP 8 — Apply SAME SciBERT setup to Medical
# Same SciBERT model, same CS-fitted scaler, same selected dimensions
# ============================================================

if os.path.exists(MED_SCIBERT_FILE):
    E_med_scibert_raw = np.load(MED_SCIBERT_FILE)
    print("Loaded existing Medical SciBERT embeddings:", E_med_scibert_raw.shape)
else:
    E_med_scibert_raw = embed_texts_scibert(
        med_docs["text_clean"].fillna("").astype(str).values,
        batch_size=16,
        max_length=512
    )
    np.save(MED_SCIBERT_FILE, E_med_scibert_raw)
    print("Saved Medical SciBERT embeddings:", MED_SCIBERT_FILE)

# Transform using CS-fitted scaler only
E_med_scibert = scibert_scaler.transform(E_med_scibert_raw)

print("========== STEP 8 SCIBERT CROSS-DOMAIN TRANSFER ==========")
print("Medical SciBERT embedding shape:", E_med_scibert.shape)
print("Selected CS-derived SciBERT dimensions:", len(selected_scibert_dim_ids))

In [ ]:
# ============================================================
# STEP 9 — SciBERT Feature Matrix for Domain B / Medical
# Same selected CS contrast dimensions
# ============================================================

X_med_scibert_selected = E_med_scibert[:, selected_scibert_dim_ids]
X_med_scibert_selected_norm = normalize(X_med_scibert_selected, norm="l2", axis=1)

print("========== STEP 9 SCIBERT FEATURE MATRIX DOMAIN B / MEDICAL ==========")
print("Selected Medical SciBERT matrix shape:", X_med_scibert_selected_norm.shape)
print("Rows/documents:", X_med_scibert_selected_norm.shape[0])
print("Selected dimensions:", X_med_scibert_selected_norm.shape[1])

In [ ]:
# ============================================================
# STEP 10 — SciBERT + UMAP + Clustering on Domain B / Medical
# Use same UMAP reducer fitted on CS
# ============================================================

Z_med_scibert_umap = scibert_umap_reducer.transform(X_med_scibert_selected_norm)

print("========== STEP 10 SCIBERT + UMAP DOMAIN B ==========")
print("Original selected Medical SciBERT shape:", X_med_scibert_selected_norm.shape)
print("UMAP-reduced Medical shape:", Z_med_scibert_umap.shape)

scibert_med_metrics, scibert_med_preds = run_clustering_suite(
    Z_med_scibert_umap,
    med_docs["label"].values,
    domain_name="Domain B / Medical",
    representation_name="SciBERT + UMAP"
)

plot_umap_2d(
    Z_med_scibert_umap,
    med_docs["label"].values,
    "SciBERT + UMAP | Domain B / Medical by true label"
)

plot_umap_2d(
    Z_med_scibert_umap,
    scibert_med_preds["KMeans"],
    "SciBERT + UMAP | Domain B / Medical by KMeans cluster"
)

In [ ]:
# ============================================================
# STEP 11 — SciBERT Evaluation
# Purity | ARI | NMI | Jaccard-like dimension stability | Direction agreement
# ============================================================

E_med_ai = E_med_scibert[ai_mask_med]
E_med_human = E_med_scibert[human_mask_med]

mean_med_emb_ai = E_med_ai.mean(axis=0)
mean_med_emb_human = E_med_human.mean(axis=0)

med_scibert_contrast_all = mean_med_emb_ai - mean_med_emb_human

scibert_stability_df = selected_scibert_features.copy()
scibert_stability_df["medical_mean_scibert_ai"] = mean_med_emb_ai[selected_scibert_dim_ids]
scibert_stability_df["medical_mean_scibert_human"] = mean_med_emb_human[selected_scibert_dim_ids]
scibert_stability_df["medical_contrast_ai_minus_human"] = med_scibert_contrast_all[selected_scibert_dim_ids]

scibert_stability_df["cs_sign"] = np.sign(scibert_stability_df["contrast_ai_minus_human"])
scibert_stability_df["medical_sign"] = np.sign(scibert_stability_df["medical_contrast_ai_minus_human"])
scibert_stability_df["same_direction"] = scibert_stability_df["cs_sign"] == scibert_stability_df["medical_sign"]

# Jaccard-style overlap:
# Compare CS top dimensions with Medical top dimensions computed only for evaluation
med_scibert_contrast_df = pd.DataFrame({
    "dimension_id": np.arange(E_med_scibert.shape[1]),
    "medical_contrast_ai_minus_human": med_scibert_contrast_all
})

med_top_ai_dims = set(
    med_scibert_contrast_df
    .sort_values("medical_contrast_ai_minus_human", ascending=False)
    .head(TOP_N_AI_DIMS)["dimension_id"]
)

med_top_human_dims = set(
    med_scibert_contrast_df
    .sort_values("medical_contrast_ai_minus_human", ascending=True)
    .head(TOP_N_HUMAN_DIMS)["dimension_id"]
)

cs_top_ai_dims = set(top_ai_scibert_dims["dimension_id"])
cs_top_human_dims = set(top_human_scibert_dims["dimension_id"])

dimension_jaccard_ai = jaccard_score_sets(cs_top_ai_dims, med_top_ai_dims)
dimension_jaccard_human = jaccard_score_sets(cs_top_human_dims, med_top_human_dims)

overall_scibert_direction_agreement = scibert_stability_df["same_direction"].mean()
ai_scibert_direction_agreement = scibert_stability_df.loc[
    scibert_stability_df["feature_group"] == "AI", "same_direction"
].mean()
human_scibert_direction_agreement = scibert_stability_df.loc[
    scibert_stability_df["feature_group"] == "Human", "same_direction"
].mean()

scibert_stability_summary = pd.DataFrame([{
    "representation": "SciBERT + UMAP",
    "dimension_jaccard_ai": dimension_jaccard_ai,
    "dimension_jaccard_human": dimension_jaccard_human,
    "overall_direction_agreement": overall_scibert_direction_agreement,
    "ai_dimension_direction_agreement": ai_scibert_direction_agreement,
    "human_dimension_direction_agreement": human_scibert_direction_agreement,
    "coherence_note": "NPMI word coherence is not applicable to SciBERT embedding dimensions."
}])

print("========== SCIBERT STABILITY SUMMARY ==========")
display(scibert_stability_summary)

display(scibert_stability_df.head())

scibert_stability_df.to_csv(
    os.path.join(SCIBERT_OUTPUT_DIR, "step11_scibert_directional_stability.csv"),
    index=False
)

scibert_stability_summary.to_csv(
    os.path.join(SCIBERT_OUTPUT_DIR, "step11_scibert_stability_summary.csv"),
    index=False
)

In [ ]:
# ============================================================
# STEP 12 — Final Analysis
# Cross-domain stability of AI-writing signals
# Compare TF-IDF + UMAP vs SciBERT + UMAP
# ============================================================

all_clustering_metrics = pd.concat([
    tfidf_cs_metrics,
    tfidf_med_metrics,
    scibert_cs_metrics,
    scibert_med_metrics
], ignore_index=True)

print("========== FINAL CLUSTERING COMPARISON ==========")
display(all_clustering_metrics)

# KMeans-only summary because your original pipeline used KMeans as main clustering
kmeans_summary = all_clustering_metrics[
    all_clustering_metrics["algorithm"] == "KMeans"
].copy()

print("========== FINAL KMEANS SUMMARY ==========")
display(kmeans_summary)

final_stability_summary = pd.concat([
    tfidf_stability_summary,
    scibert_stability_summary
], ignore_index=True, sort=False)

print("========== FINAL STABILITY SUMMARY ==========")
display(final_stability_summary)

all_clustering_metrics.to_csv(
    os.path.join(FINAL_OUTPUT_DIR, "final_all_clustering_metrics.csv"),
    index=False
)

kmeans_summary.to_csv(
    os.path.join(FINAL_OUTPUT_DIR, "final_kmeans_metrics_summary.csv"),
    index=False
)

final_stability_summary.to_csv(
    os.path.join(FINAL_OUTPUT_DIR, "final_stability_summary.csv"),
    index=False
)

# Save selected feature artifacts
selected_tfidf_vocab.to_csv(
    os.path.join(FINAL_OUTPUT_DIR, "final_selected_tfidf_words.csv"),
    index=False
)

selected_scibert_features.to_csv(
    os.path.join(FINAL_OUTPUT_DIR, "final_selected_scibert_dimensions.csv"),
    index=False
)

print("Saved final outputs to:", FINAL_OUTPUT_DIR)

print("""
FINAL INTERPRETATION TEMPLATE:

1. TF-IDF + UMAP:
   - Uses CS-only TF-IDF fitting.
   - Computes AI-Human contrast in CS.
   - Selects top 100 AI-dominant and top 100 Human-dominant words.
   - Applies the same CS-derived vocabulary to Medical.
   - Applies UMAP before clustering in both CS and Medical.
   - Reports Purity, ARI, NMI, Jaccard, directional agreement, and NPMI coherence.

2. SciBERT + UMAP:
   - Uses SciBERT embeddings instead of TF-IDF.
   - Computes AI-Human contrast over CS SciBERT embedding dimensions.
   - Selects top 100 AI-dominant and top 100 Human-dominant embedding dimensions.
   - Applies the same selected dimensions to Medical.
   - Applies UMAP before clustering in both CS and Medical.
   - Reports Purity, ARI, NMI, dimension-overlap Jaccard, and directional agreement.
   - Word-level NPMI coherence is not applicable to SciBERT dimensions.

3. Cross-domain conclusion:
   - Compare CS and Medical KMeans results.
   - If Medical remains strong, the AI-writing signal transfers across domains.
   - Compare TF-IDF vs SciBERT to see whether lexical contrast or semantic embedding contrast is more stable.
""")